# 모듈 6. Memory PII 보호

이 노트북은 Agent Memory에 대화 내용을 저장하기 전에 개인정보를 탐지하고 마스킹하는 방법을 실습합니다.

## 학습 목표

- Memory에 개인정보가 저장될 때 생기는 위험을 설명한다.
- 전화번호, 주민등록번호, 이메일 등 PII 패턴을 탐지한다.
- 저장 전 마스킹을 적용하는 Memory 래퍼를 구현한다.
- 삭제 요청 처리와 보관 정책의 필요성을 이해한다.

## 왜 이 보안 기능을 배워야 할까?

Memory는 Agent가 사용자를 더 잘 돕도록 대화 맥락을 저장합니다. 하지만 전화번호, 주민등록번호, 계좌번호, 이메일 같은 개인정보를 그대로 저장하면 유출 사고와 규제 리스크가 생깁니다.

이번 A/B 테스트는 같은 사용자 메시지를 사용합니다.

- **A안: 보안 없음** - 메시지를 그대로 저장한다.
- **B안: 보안 적용** - 저장 전에 PII를 마스킹하고 삭제 요청을 처리한다.

In [ ]:
import re
from dataclasses import dataclass, field
from datetime import datetime


messages = [
    "안녕하세요. 제 이름은 김민수이고 전화번호는 010-1234-5678입니다.",
    "주민등록번호는 900101-1234567이고 이메일은 minsu@example.com입니다.",
    "지난 상담 내용을 다음에도 기억해 주세요.",
]

## A안. Memory에 원문 그대로 저장하기

개발 단계에서는 편해 보이지만, 운영 환경에서는 원문 저장이 큰 위험이 됩니다.

In [ ]:
@dataclass
class RawMemory:
    records: list[str] = field(default_factory=list)

    def save(self, message: str) -> None:
        self.records.append(message)

    def load(self) -> list[str]:
        return self.records


raw_memory = RawMemory()
for message in messages:
    raw_memory.save(message)

for record in raw_memory.load():
    print(record)

## B안. 저장 전 PII 탐지와 마스킹 적용

아래 예시는 정규식 기반 마스킹입니다. 실제 서비스에서는 국가별 개인정보 형식, 업무 도메인별 민감정보, 사용자의 동의 범위, 삭제 정책까지 함께 설계합니다.

In [ ]:
PII_REGEX = {
    "rrn": re.compile(r"(?<!\d)\d{6}-\d{7}(?!\d)"),
    "phone": re.compile(r"(?<!\d)01[016789]-\d{3,4}-\d{4}(?!\d)"),
    "email": re.compile(
        r"(?<![A-Za-z0-9._%+-])[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}(?![A-Za-z0-9.-])"
    ),
}


def mask_pii(text: str) -> str:
    masked = PII_REGEX["rrn"].sub("[주민등록번호 마스킹]", text)
    masked = PII_REGEX["phone"].sub("[전화번호 마스킹]", masked)
    masked = PII_REGEX["email"].sub("[이메일 마스킹]", masked)
    return masked


@dataclass
class MemoryRecord:
    user_id: str
    content: str
    saved_at: str
    pii_masked: bool


@dataclass
class ProtectedMemory:
    records: list[MemoryRecord] = field(default_factory=list)

    def save(self, user_id: str, message: str) -> None:
        masked = mask_pii(message)
        self.records.append(
            MemoryRecord(
                user_id=user_id,
                content=masked,
                saved_at=datetime.now().isoformat(timespec="seconds"),
                pii_masked=masked != message,
            )
        )

    def load(self, user_id: str) -> list[MemoryRecord]:
        return [record for record in self.records if record.user_id == user_id]

    def forget_user(self, user_id: str) -> int:
        before = len(self.records)
        self.records = [record for record in self.records if record.user_id != user_id]
        return before - len(self.records)

In [ ]:
protected_memory = ProtectedMemory()
for message in messages:
    protected_memory.save("user-123", message)

for record in protected_memory.load("user-123"):
    print(record.content, "| pii_masked=", record.pii_masked)

deleted_count = protected_memory.forget_user("user-123")
print(f"삭제 요청 처리: {deleted_count}건 삭제")
print("삭제 후 기록:", protected_memory.load("user-123"))

## 확장 아이디어: 암호화 저장

이 노트북에서는 의존성 없이 마스킹만 다룹니다. 운영 환경에서는 다음을 추가로 고려합니다.

- 저장소 암호화: 예를 들어 `cryptography.Fernet` 같은 검증된 라이브러리 사용
- 키 관리: 애플리케이션 코드와 암호화 키 분리
- 보관 기간: 필요 이상 오래 저장하지 않기
- 삭제 요청: 사용자 단위로 추적하고 삭제할 수 있는 구조 유지
- 감사 로그: 누가 언제 조회/삭제했는지 기록

## 비교 정리

- A안은 전화번호, 주민등록번호, 이메일을 원문 그대로 저장했습니다.
- B안은 저장 전에 PII를 마스킹하고 사용자 삭제 요청을 처리할 수 있게 만들었습니다.
- Memory 보안은 "나중에 로그에서 지우기"가 아니라 "저장되기 전에 줄이기"가 핵심입니다.

## 실습 과제

1. 계좌번호 패턴을 추가해 마스킹해보세요.
2. `ProtectedMemory`에 보관 기간이 지난 기록을 삭제하는 `expire_old_records()` 메서드를 추가해보세요.